# Mini-hackathon: informal settlements in Metro Manila

*SSTA workshop, 27 November 2026.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adb-sr/sr-client/blob/main/notebooks/hackathon/informal_settlements/hackathon_informal_settlements.ipynb)

**The question for your team:** Would you use super-resolved Sentinel-2 to decide **where to send survey teams** to find and verify informal settlements, and how?

There is no leaderboard. Your result is a **verdict** for a real decision in your agency:
*deploy*, *pilot* or *don't use*, backed by the numbers from this notebook. "Super-resolution
doesn't help here" is a perfectly good result.

**How this notebook works.** All model predictions were computed before the workshop, so
everything runs in a few seconds on a normal computer. You do not need a GPU. You may use any
AI assistant you like to change the code.

| Part | What you do | Time |
|---|---|---|
| 0–2 | Load the data, see what the model does on each input | 10 min |
| 3 | Look at examples, including failures | 10 min |
| 4 | Write down **your** scenario | 15 min |
| 5–6 | The decision cell: does super-resolution change your decision? | main work |
| 7–8 | Optional: improve the baseline, or try your own area | if time |
| 9 | Numbers for your slide | 15 min |

*Data: expert-drawn informal-settlement outlines for 40 sample tiles (about 20 km²) in Manila South and Quezon City.*

## 0. Setup

Run the cells one by one (Shift+Enter), each once. The first one holds the helper code; you do not need to read it. Most cells show **controls** (menus, sliders, buttons, text fields): they act at once, so you do not need to run the cell again after changing them. Your answers are saved as you type. If a control seems to do nothing, run its cell again.

In [ ]:
# @title Helper functions (run this cell; you do not need to read it) { display-mode: "form" }
# Helpers for the SSTA mini-hackathon notebooks.
# This file is pasted into a hidden cell of each notebook by build.py, so every notebook
# is self-contained. Edit it here, then re-run build.py.

import json
import os
import shutil
import sys
import time
import urllib.request
import zipfile
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch, Rectangle
from PIL import Image

# --------------------------------------------------------------------------- look

COLOURS = {"s2": "#9e9e9e", "stage1": "#8fd3b6", "stage2": "#1f8a70", "hr": "#007db7",
           "random": "#d9d9d9", "best": "#4d4d4d", "perfect": "#4d4d4d"}
SHORT = {"s2": "Sentinel-2", "stage1": "SR stage 1", "stage2": "SR stage 2",
         "hr": "Real high-res", "s2_10m": "Sentinel-2 as delivered (10 m)"}
HIT_C, MISS_C = "#1a9850", "#d73027"
plt.rcParams.update({"figure.dpi": 100, "font.size": 10, "axes.titlesize": 10,
                     "axes.spines.top": False, "axes.spines.right": False})

# --------------------------------------------------------------------------- data

DATA_ROOT = Path("/content/hackathon_data") if Path("/content").exists() else Path("hackathon_data")


def _fetch(url, dest, token=None):
    """Download url to dest with a simple progress line. url may also be a local path."""
    if not str(url).startswith(("http://", "https://")):
        shutil.copy(url, dest)
        return
    req = urllib.request.Request(url, headers={"User-Agent": "ssta-hackathon"})
    if token:
        req.add_header("Authorization", f"Bearer {token}")
    with urllib.request.urlopen(req, timeout=60) as r, open(dest, "wb") as f:
        total = int(r.headers.get("Content-Length") or 0)
        done, t0, last = 0, time.time(), 0.0
        while True:
            chunk = r.read(1 << 20)
            if not chunk:
                break
            f.write(chunk)
            done += len(chunk)
            if time.time() - last > 2:
                last = time.time()
                pct = f"{done / total:.0%}" if total else ""
                print(f"\r  {done / 1e6:,.0f} MB {pct}   ", end="")
        print(f"\r  {done / 1e6:,.0f} MB in {time.time() - t0:.0f} s   ")


def get_task(task, base_url, extras=(), token=None):
    """Download and unzip <base_url>/<task>.zip (and optional extra zips) once; return a Task."""
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    for name, required in [(task, True)] + [(e, False) for e in extras]:
        marker = DATA_ROOT / f".{name}.done"
        if marker.exists():
            continue
        url = f"{str(base_url).rstrip('/')}/{name}.zip"
        tmp = DATA_ROOT / f"{name}.zip"
        print(f"Downloading {name}.zip ...")
        try:
            _fetch(url, tmp, token)
            with zipfile.ZipFile(tmp) as z:
                z.extractall(DATA_ROOT)
            tmp.unlink()
            marker.touch()
        except Exception as e:  # extras are optional (e.g. restricted high-res images)
            if required:
                raise RuntimeError(f"Could not get {url}: {e}. Ask a facilitator.") from e
            print(f"  (skipped {name}: {e}; the notebook works without it)")
    return Task(DATA_ROOT / task)


class Task:
    def __init__(self, root):
        self.root = Path(root)
        self.meta = json.loads((self.root / "meta.json").read_text())
        self.inputs = [i["key"] for i in self.meta["inputs"]]
        self.labels = {i["key"]: i["label"] for i in self.meta["inputs"]}
        self.scenes = pd.DataFrame(self.meta["scenes"]).set_index("id")
        self.px = float(self.meta.get("pixel_m", 2.5))
        first = self.scenes.index[0]
        self.has_image = {k: (self.root / "img" / first / f"{k}.jpg").exists() for k in self.inputs}

    def __repr__(self):
        return f"<{self.meta['title']}: {len(self.scenes)} scenes, inputs {self.inputs}>"

    @lru_cache(maxsize=48)
    def _npz(self, sid):
        return np.load(self.root / "arr" / f"{sid}.npz")

    @lru_cache(maxsize=32)
    def arr(self, sid, key):
        """One array of a scene (decompressed once and kept; do not change it in place)."""
        return self._npz(sid)[key]

    def keys(self, sid):
        return self._npz(sid).files

    def img(self, sid, key, pre=False):
        """Display RGB (uint8) on the 2.5 m grid. key = an input or 's2_10m' (shown as 10 m blocks)."""
        suffix = "_pre" if pre else ""
        if key == "s2_10m":
            im = Image.open(self.root / "img" / sid / f"s2_10m{suffix}.png").convert("RGB")
            h, w = int(self.scenes.loc[sid, "h"]), int(self.scenes.loc[sid, "w"])
            return np.asarray(im.resize((w, h), Image.NEAREST))
        p = self.root / "img" / sid / f"{key}{suffix}.jpg"
        if not p.exists():
            return None
        return np.asarray(Image.open(p).convert("RGB"))

    @lru_cache(maxsize=24)
    def _view_u8(self, sid, key, stretch, pre):
        im = _view(self.img(sid, key, pre=pre), stretch)
        return None if im is None else (im * 255).round().astype(np.uint8)

    def view(self, sid, key, stretch="auto", pre=False):
        """The image as shown (contrast-stretched once per image and kept), floats 0-1."""
        im = self._view_u8(sid, key, stretch, pre)
        return None if im is None else im / 255.0


def autostretch(rgb, lo=2, hi=98):
    """Per-image contrast stretch for viewing only (the models saw the original values)."""
    if rgb is None:
        return None
    x = rgb.astype(np.float32)
    out = np.empty_like(x)
    for c in range(3):
        a, b = np.percentile(x[..., c], [lo, hi])
        out[..., c] = np.clip((x[..., c] - a) / max(b - a, 1), 0, 1)
    return out


def _view(rgb, stretch):
    return autostretch(rgb) if stretch == "auto" else (None if rgb is None else rgb / 255.0)


def _outline(mask):
    m = mask.astype(bool)
    edge = np.zeros_like(m)
    edge[1:, :] |= m[1:, :] != m[:-1, :]
    edge[:, 1:] |= m[:, 1:] != m[:, :-1]
    return edge & m


def _overlay(ax, mask, colour, alpha=1.0):
    rgba = np.zeros(mask.shape + (4,))
    rgba[mask] = list(plt.matplotlib.colors.to_rgb(colour)) + [alpha]
    ax.imshow(rgba, interpolation="nearest")


def _blank(ax, text="not available"):
    ax.text(0.5, 0.5, text, ha="center", va="center", color="#777", transform=ax.transAxes)
    ax.set_xticks([]), ax.set_yticks([])


# --------------------------------------------------------------------------- decisions

def expected_hits(score, hit, k):
    """Hits among the top-k by score. Ties at the cut-off are shared out at their hit rate,
    so a model that gives many places the same score is not lucky or unlucky by chance."""
    score = np.asarray(score, float)
    hit = np.asarray(hit, bool)
    k = min(int(k), len(score))
    if k == 0:
        return 0.0
    v = np.sort(score)[::-1][k - 1]
    above, tied = score > v, score == v
    return float(hit[above].sum() + (k - above.sum()) * hit[tied].mean())


def top_k(score, k, seed=0):
    """Indices of the top-k (ties broken at random, reproducibly)."""
    rng = np.random.default_rng(seed)
    jitter = rng.random(len(score)) * 1e-9
    return np.argsort(-(np.asarray(score, float) + jitter))[:k]


def decide(units, preds, hit, budget, cost_wasted=1.0, cost_missed=1.0, labels=None):
    """Score the decision 'visit the top <budget> places' for every input.

    units: DataFrame, one row per place.  preds: {input: array (n_models, n_places)}.
    hit: bool array, which places are worth visiting (from the reference labels).
    Returns a DataFrame with one row per input plus 'random' and 'perfect'."""
    hit = np.asarray(hit, bool)
    n_hit, n = int(hit.sum()), len(hit)
    k = min(int(budget), n)
    rows = []

    def row(name, hits_per_model):
        h = np.asarray(hits_per_model, float)
        m = h.mean()
        rows.append({
            "input": name,
            "hits": round(m, 1),
            "range over models": f"{h.min():.0f}–{h.max():.0f}" if len(h) > 1 else "",
            "hit rate": f"{m / k:.0%}" if k else "",
            "wasted visits": round(k - m, 1),
            "missed places": round(n_hit - m, 1),
            "cost": round((k - m) * cost_wasted + (n_hit - m) * cost_missed, 1),
        })

    for inp, p in preds.items():
        row(inp, [expected_hits(pm, hit, k) for pm in np.atleast_2d(p)])
    row("random", [k * n_hit / max(n, 1)])
    row("perfect", [min(k, n_hit)])
    out = pd.DataFrame(rows).set_index("input")
    if labels:
        out.index = [labels.get(i, i) for i in out.index]
    print(f"{n:,} places, {n_hit:,} of them are hits ({n_hit / max(n, 1):.0%}); budget {k}.")
    return out


def plot_decision(res, budget, title=""):
    names = list(res.index)
    keys = [next((k for k, v in SHORT.items() if v == nm or k == nm), nm) for nm in names]
    cols = [COLOURS.get(k, "#999") for k in keys]
    fig, ax = plt.subplots(figsize=(7, 0.45 * len(names) + 1))
    y = np.arange(len(names))[::-1]
    ax.barh(y, res["hits"], color=cols)
    for yi, v in zip(y, res["hits"]):
        ax.text(v + budget * 0.01, yi, f"{v:g}", va="center")
    ax.set_yticks(y, names)
    ax.set_xlim(0, budget * 1.12)
    ax.axvline(budget, color="#bbb", lw=1, ls="--")
    ax.set_xlabel(f"hits among the {budget} places you would visit")
    ax.set_title(title, loc="left")
    plt.tight_layout()
    return fig


def sweep(preds, hit, budgets=(5, 10, 20, 30, 50, 75, 100), labels=None, title=""):
    """Hit rate for a range of budgets: does the advantage of one input hold for small and large budgets?"""
    hit = np.asarray(hit, bool)
    fig, ax = plt.subplots(figsize=(7, 3.6))
    budgets = [b for b in budgets if b <= len(hit)]
    for inp, p in preds.items():
        r = [np.mean([expected_hits(pm, hit, b) for pm in np.atleast_2d(p)]) / b for b in budgets]
        ax.plot(budgets, r, "o-", color=COLOURS.get(inp, "#999"), label=(labels or {}).get(inp, inp), lw=2)
    ax.plot(budgets, [hit.mean()] * len(budgets), "--", color="#999", label="random")
    ax.plot(budgets, [min(b, hit.sum()) / b for b in budgets], ":", color=COLOURS["best"], label="perfect")
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("budget (places visited)")
    ax.set_ylabel("share of visits that are hits")
    ax.set_title(title, loc="left")
    ax.legend(frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(1, 0.5))
    plt.tight_layout()
    plt.show()


def overlap_table(preds, k, labels=None, seed=0):
    """Share of the top-k places that two inputs have in common (first model of each)."""
    picks = {i: set(top_k(np.atleast_2d(p)[0], k, seed)) for i, p in preds.items()}
    names = list(picks)
    m = pd.DataFrame(index=names, columns=names, dtype=float)
    for a in names:
        for b in names:
            m.loc[a, b] = len(picks[a] & picks[b]) / max(k, 1)
    if labels:
        m.index = [labels.get(i, i) for i in names]
        m.columns = m.index
    try:
        return m.style.format("{:.0%}").background_gradient(cmap="Greens", vmin=0, vmax=1)
    except Exception:  # no jinja2: plain table
        return m.map(lambda v: f"{v:.0%}") if hasattr(m, "map") else m.applymap(lambda v: f"{v:.0%}")


def slide_numbers(res, scenario, budget, task_title, fname="my_slide_chart.png"):
    """Print the numbers for the team slide and save the bar chart as a PNG."""
    print(f"TASK: {task_title}")
    for k, v in scenario.items():
        print(f"{k.replace('_', ' ').upper()}: {v}")
    print()
    print(res[["hits", "range over models", "hit rate", "missed places", "cost"]].to_string())
    fig = plot_decision(res, budget, f"{scenario.get('decision', '')}")
    fig.savefig(fname, dpi=200, bbox_inches="tight")
    print(f"\nSaved {fname} (in Colab: Files panel on the left, right-click > Download).")


# --------------------------------------------------------------------------- informal settlements

def ifs_f1_table(task, thr=0.5):
    """Pixel F1 of the informal-settlement class per input, mean and spread over the task models."""
    acc = {}
    for sid in task.scenes.index:
        lab, val = task.arr(sid, "label"), task.arr(sid, "valid")
        v = val & (lab != 255)
        t = (lab == 1) & v
        for inp in task.inputs:
            p = task.arr(sid, f"prob_{inp}") >= thr * 255
            tp = (p & t).sum((1, 2))
            fp = (p & ~t & v).sum((1, 2))
            fn = (~p & t).sum((1, 2))
            a = acc.setdefault(inp, np.zeros((3, len(tp))))
            a += [tp, fp, fn]
    paper = {r["input"]: r.get("paper") for r in task.meta["metric_table"]}
    rows = []
    for inp, (tp, fp, fn) in acc.items():
        f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1) * 100
        rows.append({"input": task.labels[inp], "F1 % (mean)": round(f1.mean(), 1),
                     "lowest model": round(f1.min(), 1), "highest model": round(f1.max(), 1),
                     "paper": paper.get(inp)})
    return pd.DataFrame(rows).set_index("input")


def ifs_units(task, cell_m=100, thr=0.5, min_valid=0.8):
    """Cut every tile into square blocks of cell_m metres. For each block: the true informal share
    and, per input and task model, the share of the block the model flags as informal."""
    c = max(4, int(round(cell_m / task.px)))
    rows, preds = [], {inp: [] for inp in task.inputs}
    for sid in task.scenes.index:
        lab, val = task.arr(sid, "label"), task.arr(sid, "valid") & (task.arr(sid, "label") != 255)
        H, W = lab.shape
        ny, nx = H // c, W // c
        if ny == 0 or nx == 0:
            continue
        def blocks(a):
            return a[..., :ny * c, :nx * c].reshape(a.shape[:-2] + (ny, c, nx, c)).sum((-3, -1))
        nv = blocks(val.astype(np.float32))
        ninf = blocks(((lab == 1) & val).astype(np.float32))
        for inp in task.inputs:
            p = (task.arr(sid, f"prob_{inp}") >= thr * 255) & val
            preds[inp].append((blocks(p.astype(np.float32)) / np.maximum(nv, 1)).reshape(p.shape[0], -1))
        for iy in range(ny):
            for ix in range(nx):
                rows.append({"scene": sid, "group": task.scenes.loc[sid, "group"], "y0": iy * c,
                             "x0": ix * c, "size": c, "valid": nv[iy, ix] / c / c,
                             "truth": ninf[iy, ix] / max(nv[iy, ix], 1)})
    units = pd.DataFrame(rows)
    preds = {k: np.concatenate(v, axis=1) for k, v in preds.items()}
    keep = units["valid"].values >= min_valid
    units = units[keep].reset_index(drop=True)
    preds = {k: v[:, keep] for k, v in preds.items()}
    print(f"{len(units):,} blocks of {c * task.px:.0f} m x {c * task.px:.0f} m in {units.scene.nunique()} tiles")
    return units, preds


def ifs_show(task, sid, inputs=None, window=None, stretch="auto", model=None, thr=0.5,
             box=None, title=None):
    """Top row: the images. Bottom row: what the task model finds (orange) and the reference outline (black)."""
    inputs = inputs or ["s2_10m"] + task.inputs
    lab = task.arr(sid, "label")
    y0, x0, s = window or (0, 0, max(lab.shape))
    sl = (slice(y0, y0 + s), slice(x0, x0 + s))
    ref_edge = _outline(lab == 1)[sl]
    fig, axes = plt.subplots(2, len(inputs), figsize=(2.6 * len(inputs), 5.6), squeeze=False)
    for j, key in enumerate(inputs):
        ax, bx = axes[0, j], axes[1, j]
        im = task.view(sid, key, stretch)
        if im is None:
            _blank(ax, "image not\ndownloaded")
        else:
            ax.imshow(im[sl], interpolation="nearest")
        ax.set_title(SHORT.get(key, key))
        pkey = "s2" if key == "s2_10m" else key
        if key == "s2_10m":
            _blank(bx, "(the model sees the\nupsampled version)")
        else:
            prob = task.arr(sid, f"prob_{pkey}")
            p = (prob[model] if model is not None else prob.mean(0)) / 255.0
            bx.imshow(np.ones(p[sl].shape + (3,)))
            _overlay(bx, p[sl] >= thr, "#e6550d", 0.85)
            _overlay(bx, ref_edge, "black")
            if box:
                for b in (ax, bx):
                    b.add_patch(Rectangle((box[1] - x0 - .5, box[0] - y0 - .5), box[2], box[2],
                                          fill=False, ec="#d73027", lw=2))
        for b in (ax, bx):
            b.set_xticks([]), b.set_yticks([])
    axes[1, 0].set_ylabel("model output")
    fig.suptitle(title or f"{sid}  ({task.scenes.loc[sid, 'group']})", x=0.01, ha="left")
    fig.legend(handles=[Patch(color="#e6550d", label="flagged as informal (mean of models)"),
                        Patch(fc="white", ec="black", label="reference outline")],
               loc="lower center", ncol=2, frameon=False)
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    plt.show()


def ifs_gallery(task, units, idx, key="stage2", stretch="auto", ctx=128, title="", hit=None):
    """Thumbnails of chosen blocks (with some context), framed green (hit) or red (not a hit)."""
    idx = list(idx)
    n = len(idx)
    cols = min(n, 8)
    rws = int(np.ceil(n / cols)) if n else 1
    fig, axes = plt.subplots(rws, cols, figsize=(1.9 * cols, 2.1 * rws), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, idx):
        u = units.iloc[i]
        im = task.view(u.scene, key, stretch)
        H, W = im.shape[:2]
        cy, cx = u.y0 + u["size"] // 2, u.x0 + u["size"] // 2
        y0 = int(np.clip(cy - ctx // 2, 0, max(H - ctx, 0)))
        x0 = int(np.clip(cx - ctx // 2, 0, max(W - ctx, 0)))
        ax.imshow(im[y0:y0 + ctx, x0:x0 + ctx])
        _overlay(ax, _outline(task.arr(u.scene, "label") == 1)[y0:y0 + ctx, x0:x0 + ctx], "black")
        good = bool(hit[i]) if hit is not None else None
        ax.add_patch(Rectangle((u.x0 - x0 - .5, u.y0 - y0 - .5), u["size"], u["size"], fill=False,
                               ec=HIT_C if good else MISS_C, lw=2.5))
        ax.set_title(f"true {u.truth:.0%}", fontsize=8)
    fig.suptitle(title, x=0.01, ha="left")
    plt.tight_layout()
    plt.show()


# --------------------------------------------------------------------------- land cover (Hanoi)

LC_NAMES = ["building", "road", "water", "tree", "low vegetation", "cropland", "bare soil or sand"]
LC_COLOURS = np.array([[215, 48, 39], [99, 99, 99], [43, 131, 186], [26, 120, 50], [166, 217, 106],
                       [240, 210, 80], [191, 129, 45], [255, 255, 255]], np.uint8) / 255.0


def lc_metric_table(task):
    """IoU per class and mean IoU per input, pooled over all tiles (pixel by pixel)."""
    cm = {inp: np.zeros((7, 7), np.int64) for inp in task.inputs}
    for sid in task.scenes.index:
        v = task.arr(sid, "valid")
        t = task.arr(sid, "label")[v].astype(np.int64)
        for inp in task.inputs:
            p = task.arr(sid, f"pred_{inp}")[v].astype(np.int64)
            cm[inp] += np.bincount(t * 7 + p, minlength=49).reshape(7, 7)
    paper = {r["input"]: r.get("paper") for r in task.meta["metric_table"]}
    rows = []
    for inp, c in cm.items():
        iou = np.diag(c) / np.maximum(c.sum(0) + c.sum(1) - np.diag(c), 1)
        rows.append({"input": task.labels[inp], **{n: round(v, 2) for n, v in zip(LC_NAMES, iou)},
                     "mean (mIoU)": round(iou.mean(), 3), "paper mIoU": paper.get(inp)})
    return pd.DataFrame(rows).set_index("input")


def lc_units(task, cell_m=80, target=6, min_valid=0.8):
    """Cut every tile into square cells of cell_m metres. For each cell: the true share of the
    target class(es) and, per input, the share the task model labels as that class. Only
    labelled pixels count; cells less than min_valid labelled are left out."""
    c = max(4, int(round(cell_m / task.px)))
    cls = tuple(np.atleast_1d(target))
    rows, preds = [], {inp: [] for inp in task.inputs}
    for sid in task.scenes.index:
        lab, val = task.arr(sid, "label"), task.arr(sid, "valid")
        H, W = lab.shape
        ny, nx = H // c, W // c

        def blocks(a):
            return a[:ny * c, :nx * c].reshape(ny, c, nx, c).sum((1, 3)).ravel()
        nv = blocks(val.astype(np.float32))
        nt = blocks((np.isin(lab, cls) & val).astype(np.float32))
        for inp in task.inputs:
            p = np.isin(task.arr(sid, f"pred_{inp}"), cls) & val
            preds[inp].append(blocks(p.astype(np.float32)) / np.maximum(nv, 1))
        for i in range(ny * nx):
            rows.append({"scene": sid, "group": task.scenes.loc[sid, "group"], "y0": (i // nx) * c,
                         "x0": (i % nx) * c, "size": c, "valid": nv[i] / c / c,
                         "truth": nt[i] / max(nv[i], 1)})
    units = pd.DataFrame(rows)
    keep = units["valid"].values >= min_valid
    units = units[keep].reset_index(drop=True)
    preds = {k: np.concatenate(v)[keep][None, :] for k, v in preds.items()}
    print(f"{len(units):,} cells of {c * task.px:.0f} m x {c * task.px:.0f} m in "
          f"{units.scene.nunique()} tiles (cells less than {min_valid:.0%} labelled are left out)")
    return units, preds


def _lc_rgb(lab):
    return LC_COLOURS[np.where(lab == 255, 7, lab)]


def lc_show(task, sid, inputs=None, stretch="auto", target=None, box=None, title=None):
    """Top row: the images. Bottom row: the land cover the task model draws on each input; the
    last column is the expert map (white = not labelled). target: outline one class in black."""
    inputs = inputs or ["s2_10m"] + task.inputs
    cols = inputs + ["reference"]
    lab = task.arr(sid, "label")
    edge = _outline(np.isin(lab, np.atleast_1d(target))) if target is not None else None
    fig, axes = plt.subplots(2, len(cols), figsize=(2.5 * len(cols), 5.5), squeeze=False)
    for j, key in enumerate(cols):
        ax, bx = axes[0, j], axes[1, j]
        if key == "reference":
            ax.axis("off")
            bx.imshow(_lc_rgb(lab), interpolation="nearest")
            bx.set_title("expert map")
        else:
            im = task.view(sid, key, stretch)
            if im is None:
                _blank(ax, "image not\ndownloaded")
            else:
                ax.imshow(im, interpolation="nearest")
            ax.set_title(SHORT.get(key, key))
            if key == "s2_10m":
                _blank(bx, "(the model sees the\nupsampled version)")
            else:
                bx.imshow(_lc_rgb(task.arr(sid, f"pred_{key}")), interpolation="nearest")
        if edge is not None and key != "s2_10m":
            _overlay(bx, edge, "black")
        if box:
            for b in (ax, bx):
                b.add_patch(Rectangle((box[1] - .5, box[0] - .5), box[2], box[2], fill=False,
                                      ec="#d73027", lw=2))
        for b in (ax, bx):
            b.set_xticks([]), b.set_yticks([])
    axes[1, 0].set_ylabel("land cover drawn")
    s = task.scenes.loc[sid]
    fig.suptitle(title or f"{sid}  ({s.group}, 640 m across)", x=0.01, ha="left")
    fig.legend(handles=[Patch(color=LC_COLOURS[i], label=n) for i, n in enumerate(LC_NAMES)]
               + [Patch(fc="white", ec="#999", label="not labelled")],
               loc="lower center", ncol=8, frameon=False, fontsize=8)
    plt.tight_layout(rect=(0, 0.05, 1, 1))
    plt.show()


def lc_gallery(task, units, idx, key="stage2", stretch="auto", ctx=96, title="", hit=None,
               target=6):
    """Thumbnails of chosen cells (with some context), the target class outlined in black from
    the expert map, framed green (hit) or red (not a hit)."""
    idx = list(idx)
    cols = min(len(idx), 8) or 1
    rws = int(np.ceil(len(idx) / cols)) or 1
    fig, axes = plt.subplots(rws, cols, figsize=(1.9 * cols, 2.1 * rws), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, idx):
        u = units.iloc[i]
        im = task.view(u.scene, key, stretch)
        H, W = im.shape[:2]
        ctx_ = max(ctx, int(u["size"]) + 16)
        cy, cx = u.y0 + u["size"] // 2, u.x0 + u["size"] // 2
        y0 = int(np.clip(cy - ctx_ // 2, 0, max(H - ctx_, 0)))
        x0 = int(np.clip(cx - ctx_ // 2, 0, max(W - ctx_, 0)))
        ax.imshow(im[y0:y0 + ctx_, x0:x0 + ctx_])
        lab = task.arr(u.scene, "label")[y0:y0 + ctx_, x0:x0 + ctx_]
        _overlay(ax, _outline(np.isin(lab, np.atleast_1d(target))), "black")
        good = bool(hit[i]) if hit is not None else None
        ax.add_patch(Rectangle((u.x0 - x0 - .5, u.y0 - y0 - .5), u["size"], u["size"], fill=False,
                               ec=HIT_C if good else MISS_C, lw=2.5))
        ax.set_title(f"true {u.truth:.0%}", fontsize=8)
    fig.suptitle(title, x=0.01, ha="left")
    plt.tight_layout()
    plt.show()


# --------------------------------------------------------------------------- field boundaries

def ai4_units(task, cell_m=500, small_m2=None, min_px=16, cache=True):
    """Cut every tile into square cells of cell_m metres and count, per cell, the fields whose
    centre lies in it: reference fields and the fields each model draws (one per seed).
    small_m2: count only fields smaller than this (None = all fields)."""
    c = max(8, int(round(cell_m / task.px)))
    key = f"units_{c}_{small_m2}_{min_px}.npz"
    cpath = task.root / key
    seeds = range(len(task.meta.get("models", [0, 1, 2])))
    if cache and cpath.exists():
        z = np.load(cpath, allow_pickle=True)
        units = pd.DataFrame(z["units"].tolist())
        return units, {k[5:]: z[k] for k in z.files if k.startswith("pred_")}
    a_px = task.px * task.px
    rows, preds = [], {inp: [] for inp in task.inputs}
    t0 = time.time()
    for sid in task.scenes.index:
        H, W = int(task.scenes.loc[sid, "h"]), int(task.scenes.loc[sid, "w"])
        ny, nx = H // c, W // c
        yy, xx = np.mgrid[0:H, 0:W]

        def counts(ids, min_px_):
            ids = ids.ravel().astype(np.int64)
            n = int(ids.max()) + 1
            cnt = np.bincount(ids, minlength=n)
            cy = np.bincount(ids, weights=yy.ravel(), minlength=n) / np.maximum(cnt, 1)
            cx = np.bincount(ids, weights=xx.ravel(), minlength=n) / np.maximum(cnt, 1)
            ok = (cnt >= min_px_) & (np.arange(n) > 0)
            if small_m2:
                ok &= cnt * a_px < small_m2
            iy, ix = (cy[ok] // c).astype(int), (cx[ok] // c).astype(int)
            inside = (iy < ny) & (ix < nx)
            return np.bincount(iy[inside] * nx + ix[inside], minlength=ny * nx)

        valid = task.arr(sid, "valid")
        vfrac = valid[:ny * c, :nx * c].reshape(ny, c, nx, c).mean((1, 3)).ravel()
        ref = counts(task.arr(sid, "fields_ref"), min_px)
        for inp in task.inputs:
            preds[inp].append(np.stack([counts(task.arr(sid, f"fields_{inp}_s{s}"), min_px) for s in seeds]))
        for i in range(ny * nx):
            rows.append({"scene": sid, "group": task.scenes.loc[sid, "group"], "y0": (i // nx) * c,
                         "x0": (i % nx) * c, "size": c, "valid": vfrac[i], "truth": int(ref[i])})
        print(f"\r  {sid} done ({time.time() - t0:.0f} s)   ", end="")
    print()
    units = pd.DataFrame(rows)
    preds = {k: np.concatenate(v, axis=1).astype(float) for k, v in preds.items()}
    if cache:
        np.savez(cpath, units=np.array(units.to_dict("records"), dtype=object),
                 **{f"pred_{k}": v for k, v in preds.items()})
    return units, preds


def ai4_tile_counts(task, small_m2=None, min_px=16):
    """Number of fields per tile: reference against each input (mean of the seeds)."""
    a_px = task.px * task.px
    seeds = range(len(task.meta.get("models", [0, 1, 2])))
    rows = []
    for sid in task.scenes.index:
        def n_fields(ids):
            cnt = np.bincount(ids.ravel().astype(np.int64))
            ok = (cnt >= min_px) & (np.arange(len(cnt)) > 0)
            if small_m2:
                ok &= cnt * a_px < small_m2
            return int(ok.sum())
        r = {"tile": sid, "reference": n_fields(task.arr(sid, "fields_ref"))}
        for inp in task.inputs:
            r[task.labels[inp]] = round(np.mean([n_fields(task.arr(sid, f"fields_{inp}_s{s}")) for s in seeds]))
        rows.append(r)
    df = pd.DataFrame(rows).set_index("tile")
    tot = df.sum()
    df.loc["ALL TILES"] = tot
    return df


def ai4_show(task, sid, y0=0, x0=0, size=400, inputs=None, stretch="auto", seed=0, title=None):
    """Top row: the images with the reference field outlines (red). Bottom row: the fields each
    model draws (seed 0), outlined, over its boundary probability."""
    inputs = inputs or ["s2_10m"] + task.inputs
    sl = (slice(y0, y0 + size), slice(x0, x0 + size))
    ref = task.arr(sid, "fields_ref")[sl]
    ref_edge = _boundaries(ref)
    fig, axes = plt.subplots(2, len(inputs), figsize=(3.1 * len(inputs), 6.4), squeeze=False)
    for j, key in enumerate(inputs):
        ax, bx = axes[0, j], axes[1, j]
        im = task.view(sid, key, stretch)
        ax.imshow(im[sl], interpolation="nearest")
        _overlay(ax, ref_edge, "#d73027", 0.9)
        ax.set_title(SHORT.get(key, key))
        if key == "s2_10m":
            _blank(bx, "(each input has its\nown trained model)")
        else:
            prob = task.arr(sid, f"prob_{key}_s0")[sl] / 255.0 if f"prob_{key}_s0" in task.keys(sid) else None
            if prob is not None:
                bx.imshow(prob, cmap="gray_r", vmin=0, vmax=2.5)  # light grey: boundary probability
            f = task.arr(sid, f"fields_{key}_s{seed}")[sl]
            _overlay(bx, _boundaries(f), "#1f8a70", 1)
            n = len(np.unique(f[f > 0]))
            bx.set_title(f"{n} fields drawn here", fontsize=9)
        for b in (ax, bx):
            b.set_xticks([]), b.set_yticks([])
    nref = len(np.unique(ref[ref > 0]))
    axes[1, 0].set_ylabel("model output")
    fig.suptitle(title or f"{sid}: {size * task.px / 1000:.1f} km window, {nref} reference fields",
                 x=0.01, ha="left")
    fig.legend(handles=[Patch(color="#d73027", label="reference field outlines"),
                        Patch(color="#1f8a70", label="fields the model draws")],
               loc="lower center", ncol=2, frameon=False)
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    plt.show()


def _boundaries(ids):
    e = np.zeros(ids.shape, bool)
    e[1:, :] |= ids[1:, :] != ids[:-1, :]
    e[:, 1:] |= ids[:, 1:] != ids[:, :-1]
    return e


def ai4_gallery(task, units, idx, key="stage1", stretch="auto", title="", hit=None):
    idx = list(idx)
    cols = min(len(idx), 8) or 1
    rws = int(np.ceil(len(idx) / cols)) or 1
    fig, axes = plt.subplots(rws, cols, figsize=(1.9 * cols, 2.1 * rws), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, idx):
        u = units.iloc[i]
        sl = (slice(u.y0, u.y0 + u["size"]), slice(u.x0, u.x0 + u["size"]))
        ax.imshow(task.view(u.scene, key, stretch)[sl])
        _overlay(ax, _boundaries(task.arr(u.scene, "fields_ref")[sl]), "#d73027", 0.8)
        good = bool(hit[i]) if hit is not None else None
        ax.add_patch(Rectangle((0, 0), 1, 1, transform=ax.transAxes, fill=False,
                               ec=HIT_C if good else MISS_C, lw=5))
        ax.set_title(f"{u.truth} fields", fontsize=8)
    fig.suptitle(title, x=0.01, ha="left")
    plt.tight_layout()
    plt.show()


# ---- the notebook's controls (ipywidgets): menus, sliders and buttons that act at once ------
# Pasted after hk.py by build.py. Each part's cell is run once; its controls redraw at once.
# Parts 6 and 9 follow the decision settings of part 5. Test by setting widget values in UI.
import contextlib
import html as _html
import io

import ipywidgets as W
from IPython.display import display

ANSWERS = {}     # your team's answers; saved as soon as you type or click
UI = {}          # the controls of each part (used for testing)
S = {}           # this session: the task and the current decision (units, preds, hit, res)
REFRESH = []     # what to redraw when the decision settings of part 5 change
ANSWER_REFRESH = []   # what to redraw when an answer changes
_WIDE = W.Layout(width="600px")
_LABEL = {"description_width": "200px"}


def _buttons(width):
    return {"button_width": width, "description_width": "200px"}


def _png(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=90, bbox_inches="tight")
    plt.close(fig)
    return buf.getvalue()


def _note(text, colour="#333"):
    return f"<div style='color:{colour}; line-height:1.5'>{text}</div>"


def _pre(text):
    return f"<pre style='margin:4px 0; line-height:1.35'>{_html.escape(text)}</pre>"


def _error(exc):
    return _note(f"Something went wrong: {_html.escape(str(exc))}. Run the cell again, or ask "
                 "a facilitator.", "#b00020")


def _run(fn, *args, **kwargs):
    """Call fn with plt.show switched off. Returns (its result, PNGs of the figures it drew,
    what it printed), so the figures can go into image widgets."""
    plt.close("all")
    show, plt.show = plt.show, (lambda *a, **k: None)
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            out = fn(*args, **kwargs)
    finally:
        plt.show = show
    pngs = [_png(plt.figure(n)) for n in list(plt.get_fignums())]
    plt.close("all")
    return out, pngs, buf.getvalue().strip()


def _images(pngs):
    return [W.Image(value=p, format="png", layout=W.Layout(max_width="100%")) for p in pngs]


def _html_table(t):
    try:
        return t.to_html()
    except TypeError:
        return t.to_html(float_format="%.2f")


def _short(task):
    return {k: SHORT[k] for k in task.inputs}


def _kind(task):
    return task.meta["task"]


def _need_decision():
    if "res" not in S:
        display(W.HTML(_note("<b>Run the cell of part 5 first</b>, then run this cell again.",
                             "#b00020")))
        return False
    return True


# ---- answers that save themselves ------------------------------------------------------------

def ask(*questions, fill=None):
    """Each question is (key, text, kind[, example]): kind is "text" or a list of choices. The
    answer is saved in ANSWERS the moment it changes; nothing to run. ``fill`` adds a button
    that copies the examples into the empty fields."""
    rows, fields = [], {}
    for q in questions:
        key, text, kind = q[:3]
        example = q[3] if len(q) > 3 else ""
        saved = W.HTML()
        if kind == "text":
            field = W.Text(value=ANSWERS.get(key, ""), continuous_update=True, layout=_WIDE,
                           placeholder=f"e.g. {example}" if example else "type here")
        else:
            field = W.ToggleButtons(options=list(kind), value=ANSWERS.get(key),
                                    style={"button_width": "auto"})

        def on_change(change, key=key, saved=saved):
            ANSWERS[key] = change["new"]
            saved.value = _note("saved ✓", "#1a7f37") if change["new"] not in ("", None) else ""
            for redraw in ANSWER_REFRESH:
                redraw()
        field.observe(on_change, names="value")
        field.example = example
        fields[key] = field
        rows.append(W.VBox([W.HTML(f"<b>{text}</b>"), W.HBox([field, saved])],
                           layout=W.Layout(margin="4px 0 8px 0")))
    if fill:
        button = W.Button(description=fill, icon="clipboard", layout=W.Layout(width="300px"))

        def on_fill(_):
            for f in fields.values():
                if isinstance(f, W.Text) and not f.value.strip():
                    f.value = f.example
        button.on_click(on_fill)
        rows.insert(0, button)
    box = W.VBox(rows, layout=W.Layout(border="1px solid #007DB7", padding="8px 12px",
                                       margin="8px 0"))
    box.fields = fields
    display(box)
    return box


# ---- part 3: look around ---------------------------------------------------------------------

def _stretch_buttons():
    return W.ToggleButtons(options=[("auto contrast", "auto"), ("as delivered", "none")],
                           value="auto", description="Colours", style=_buttons("110px"),
                           tooltips=["each image stretched for viewing",
                                     "the colours the package stores"])


def explore(task):
    """Menus and sliders to move through the places; the picture follows at once."""
    kind = _kind(task)
    image = W.VBox([])
    status = W.HTML()
    stretch = _stretch_buttons()
    if kind == "ifs_manila":
        sc = task.scenes.sort_values("informal_share", ascending=False)
        tile = W.Dropdown(options=[(f"{sid}  ({s:.0%} informal)", sid)
                                   for sid, s in sc.informal_share.items()],
                          description="Tile (most informal first)", layout=_WIDE, style=_LABEL)
        n_models = len(task.meta.get("models", [])) or 6
        model = W.Dropdown(options=[("mean of the six models", None)]
                           + [(f"model {i + 1} only", i) for i in range(n_models)],
                           description="Model output", style=_LABEL)
        thr = W.FloatSlider(value=0.5, min=0.1, max=0.9, step=0.05, continuous_update=False,
                            readout_format=".2f", description="Flag a pixel above",
                            layout=_WIDE, style=_LABEL)
        controls = [tile, model, thr, stretch]

        def render():
            ifs_show(task, tile.value, model=model.value, thr=thr.value, stretch=stretch.value)
    elif kind == "hanoi_landcover":
        sort_by = W.Dropdown(options=[(n, i) for i, n in enumerate(LC_NAMES)], value=6,
                             description="Sort tiles by", layout=_WIDE, style=_LABEL)
        tile = W.Dropdown(description="Tile (most first)", layout=_WIDE, style=_LABEL)
        outline = W.Checkbox(value=True, description="Outline that class (expert map) in black",
                             layout=_WIDE, style=_LABEL)
        controls = [sort_by, tile, outline, stretch]

        def tiles(_=None):
            col = f"share_{sort_by.value}"
            sc = task.scenes.sort_values(col, ascending=False)
            tile.options = [(f"{sid}  ({v:.0%} {LC_NAMES[sort_by.value]})", sid)
                            for sid, v in sc[col].items()]
        sort_by.observe(tiles, names="value")
        tiles()

        def render():
            lc_show(task, tile.value, stretch=stretch.value,
                    target=sort_by.value if outline.value else None)
    else:
        area = W.Dropdown(options=[(f"{sid}  ({int(s.n_fields):,} fields)", sid)
                                   for sid, s in task.scenes.iterrows()],
                          value="16_cambodia", description="Area", layout=_WIDE, style=_LABEL)
        size = W.ToggleButtons(options=[("250 m", 100), ("500 m", 200), ("1 km", 400),
                                        ("2 km", 800)], value=400, description="Window",
                               style=_buttons("80px"))
        east = W.IntSlider(value=2000, min=0, max=4000, step=100, continuous_update=False,
                           description="← west | east → (m)", layout=_WIDE, style=_LABEL)
        south = W.IntSlider(value=2000, min=0, max=4000, step=100, continuous_update=False,
                            description="↑ north | south ↓ (m)", layout=_WIDE, style=_LABEL)
        seed = W.ToggleButtons(options=[("model 1", 0), ("model 2", 1), ("model 3", 2)], value=0,
                               description="Trained model (seed)", style=_buttons("80px"))
        controls = [area, size, east, south, seed, stretch]

        def limits(_=None):
            s = task.scenes.loc[area.value]
            east.max = int((int(s.w) - size.value) * task.px) // 100 * 100
            south.max = int((int(s.h) - size.value) * task.px) // 100 * 100
        area.observe(limits, names="value")
        size.observe(limits, names="value")
        limits()

        def render():
            y0, x0 = int(south.value / task.px), int(east.value / task.px)
            ai4_show(task, area.value, y0=y0, x0=x0, size=size.value, seed=seed.value,
                     stretch=stretch.value)

    def draw(_=None):
        try:
            _, pngs, _ = _run(render)
            image.children = _images(pngs)
        except Exception as exc:  # say what went wrong instead of failing silently
            status.value = _error(exc)
    for w in controls:
        w.observe(draw, names="value")
    ui = W.VBox([*controls, status, image])
    ui.controls = controls
    UI["explore"] = ui
    display(ui)
    draw()


# ---- part 4: the scenario --------------------------------------------------------------------

SCENARIO_FIELDS = [("team", "Team name"),
                   ("decision_maker", "Who decides?"),
                   ("decision", "What do they decide?"),
                   ("budget", "Budget: how many places can they act on?"),
                   ("hit", "What counts as a hit (a place worth the visit)?"),
                   ("cost_of_wasted_visit", "What does a wasted visit cost?"),
                   ("cost_of_missed_place", "What does a missed place cost?")]

EXAMPLES = {
    "ifs_manila": dict(
        team="Team A",
        decision_maker="DHSUD regional office, informal settlement upgrading programme",
        decision="Which city blocks get a field survey this quarter",
        budget="20 survey visits, one 100 m x 100 m block each",
        hit="a block where at least a quarter of the area is informal settlement",
        cost_of_wasted_visit="1 survey day for a team of 2",
        cost_of_missed_place="a settlement left out of the programme for a year: about 3 survey "
                             "days' worth"),
    "hanoi_landcover": dict(
        team="Team B",
        decision_maker="City environment department, air-quality enforcement unit",
        decision="Which 80 m x 80 m cells get a dust-control inspection this month (open construction "
                 "sites, cleared land, sand yards)",
        budget="20 inspections",
        hit="a cell where at least 20 % of the ground is bare soil or sand",
        cost_of_wasted_visit="1 inspector-day",
        cost_of_missed_place="a dust source left uncontrolled for a month: about 3 inspector-days' "
                             "worth"),
    "ai4_fields": dict(
        team="Team C",
        decision_maker="Department of Agriculture regional office, smallholder support programme",
        decision="Which 500 m x 500 m squares get an extension-worker visit to enrol small farms",
        budget="30 visits",
        hit="a square with at least 30 fields smaller than 0.25 ha",
        cost_of_wasted_visit="1 extension-worker day",
        cost_of_missed_place="~30 smallholders not enrolled this season: about 2 worker-days' "
                             "worth"),
}


def scenario(task):
    """Text fields for the team's scenario; saved as you type. The example is in grey."""
    ex = EXAMPLES[_kind(task)]
    UI["scenario"] = ask(*[(k, t, "text", ex[k]) for k, t in SCENARIO_FIELDS],
                         fill="Fill the empty fields with the example")


def scenario_dict():
    """The scenario as typed in part 4 (empty fields: '(not answered)')."""
    return {k: (ANSWERS.get(k) or "").strip() or "(not answered)" for k, _ in SCENARIO_FIELDS}


# ---- part 5: the decision cell ---------------------------------------------------------------

def _int_slider(value, lo, hi, step, text):
    return W.IntSlider(value=value, min=lo, max=hi, step=step, continuous_update=False,
                       description=text, layout=_WIDE, style=_LABEL)


def _cost(value, text):
    return W.BoundedFloatText(value=value, min=0, max=10000, step=0.5, description=text,
                              style=_LABEL)


def decision_panel(task):
    """The decision controls. Every change recomputes the hits at once, and parts 6 and 9
    follow. The places (blocks, areas or squares) are computed once per setting and kept."""
    kind = _kind(task)
    labels = _short(task)
    S.update(task=task, cache={})
    if kind == "ifs_manila":
        budget = _int_slider(20, 5, 100, 5, "Budget (blocks you survey)")
        size = W.ToggleButtons(options=[("50 m", 50), ("100 m", 100), ("200 m", 200)],
                               value=100, description="Block size", style=_buttons("80px"))
        hit_at = W.FloatSlider(value=0.25, min=0.05, max=0.9, step=0.05, continuous_update=False,
                               readout_format=".0%", layout=_WIDE, style=_LABEL,
                               description="Hit = informal share at least")
        cw, cm = _cost(1, "Cost of a wasted visit"), _cost(3, "Cost of a missed place")
        unit_controls, controls = [size], [budget, size, hit_at, cw, cm]

        def places():
            return ifs_units(task, cell_m=size.value)

        def settings():
            return {"block": f"{size.value} m", "hit": f"informal share ≥ {hit_at.value:.0%}"}

        def title():
            return f"Survey {budget.value} blocks of {size.value} m: how many are hits?"
        place_word = "blocks"
    elif kind == "hanoi_landcover":
        target = W.Dropdown(options=[(n, i) for i, n in enumerate(LC_NAMES)]
                            + [("built-up (building + road)", (0, 1)),
                               ("green (tree + low vegetation)", (3, 4))],
                            value=6, description="Land cover you look for", layout=_WIDE,
                            style=_LABEL)
        size = W.ToggleButtons(options=[("40 m", 40), ("80 m", 80), ("160 m", 160)],
                               value=80, description="Cell size", style=_buttons("80px"))
        budget = _int_slider(20, 5, 100, 5, "Budget (cells you inspect)")
        hit_at = W.FloatSlider(value=0.2, min=0.05, max=0.9, step=0.05, continuous_update=False,
                               readout_format=".0%", layout=_WIDE, style=_LABEL,
                               description="Hit = share of the cell at least")
        cw, cm = _cost(1, "Cost of a wasted visit"), _cost(3, "Cost of a missed place")
        unit_controls, controls = [target, size], [target, size, budget, hit_at, cw, cm]

        def what():
            return next(n for n, v in target.options if v == target.value)

        def places():
            return lc_units(task, cell_m=size.value, target=target.value)

        def settings():
            return {"looking for": what(), "cell": f"{size.value} m",
                    "hit": f"{what()} ≥ {hit_at.value:.0%} of the cell"}

        def title():
            return f"Inspect {budget.value} cells of {size.value} m ({what()}): how many are hits?"
        place_word = "cells"
    else:
        budget = _int_slider(30, 5, 100, 5, "Budget (squares you visit)")
        size = W.ToggleButtons(options=[("250 m", 250), ("500 m", 500), ("1 km", 1000)],
                               value=500, description="Square size", style=_buttons("80px"))
        small = W.ToggleButtons(options=[("all fields", 0), ("< 0.1 ha", 1000),
                                         ("< 0.25 ha", 2500), ("< 0.5 ha", 5000)], value=2500,
                                description="Count fields", style=_buttons("90px"))
        hit_at = _int_slider(30, 1, 150, 1, "Hit = at least this many fields")
        cw, cm = _cost(1, "Cost of a wasted visit"), _cost(2, "Cost of a missed place")
        unit_controls, controls = [size, small], [budget, size, small, hit_at, cw, cm]

        def places():
            return ai4_units(task, cell_m=size.value, small_m2=small.value or None)

        def settings():
            what = "fields" if not small.value else f"fields < {small.value / 10000:g} ha"
            return {"square": f"{size.value} m", "hit": f"≥ {hit_at.value} {what}"}

        def title():
            return f"Visit {budget.value} squares of {size.value} m: how many are hits?"
        place_word = "squares"

    status = W.HTML()
    chart = W.Image(format="png", layout=W.Layout(max_width="100%"))
    table = W.HTML()

    def compute(_=None):
        key = tuple(w.value for w in unit_controls)
        try:
            if key not in S["cache"]:
                status.value = _note(f"Cutting the map into {place_word} ... (the first time "
                                     "for a setting can take up to 20 seconds)")
                (units, preds), _, said = _run(places)
                S["cache"][key] = (units, preds, said)
            units, preds, said = S["cache"][key]
            hit = units.truth.values >= hit_at.value
            res, _, said2 = _run(decide, units, preds, hit, budget.value, cw.value, cm.value,
                                 labels=labels)
            S.update(units=units, preds=preds, hit=hit, res=res, budget=budget.value,
                     target=target.value if kind == "hanoi_landcover" else None,
                     hit_at=hit_at.value,
                     cost_wasted=cw.value, cost_missed=cm.value, title=title(),
                     settings={"budget": budget.value, **settings(),
                               "cost of a wasted visit": cw.value,
                               "cost of a missed place": cm.value},
                     place_word=place_word)
            _, pngs, _ = _run(plot_decision, res, budget.value, title())
            chart.value = pngs[0]
            table.value = res.to_html()
            status.value = _pre("\n".join(x for x in (said, said2) if x))
        except Exception as exc:  # say what went wrong instead of failing silently
            status.value = _error(exc)
            return
        for redraw in REFRESH:
            try:
                redraw()
            except Exception:  # a later part must not break the decision cell
                pass
    for w in controls:
        w.observe(compute, names="value")
    ui = W.VBox([*controls, status, chart, table])
    ui.controls = {w.description: w for w in controls}
    UI["decision"] = ui
    display(ui)
    compute()


def current():
    """The current decision of part 5: units (one row per place), preds ({input: array
    (models, places)}), hit (bool per place) and res (the table)."""
    return S["units"], S["preds"], S["hit"], S["res"]


# ---- part 5b: other budgets ------------------------------------------------------------------

def sweep_view():
    """Share of hits for budgets from 5 to 100; follows the settings of part 5."""
    if not _need_decision():
        return
    chart = W.Image(format="png", layout=W.Layout(max_width="100%"))

    def draw():
        _, pngs, _ = _run(sweep, S["preds"], S["hit"], labels=_short(S["task"]),
                          title=f"Share of visits that are hits ({S['place_word']}, "
                                f"{S['settings']['hit']})")
        chart.value = pngs[0]
    REFRESH.append(draw)
    draw()
    UI["sweep"] = chart
    display(chart)


# ---- part 6: which places, and the failures --------------------------------------------------

def _gallery(task, idx, key, title):
    fn = {"ifs_manila": ifs_gallery, "hanoi_landcover": lc_gallery, "ai4_fields": ai4_gallery}[
        _kind(task)]
    if not len(idx):
        return []
    extra = {"target": S["target"]} if _kind(task) == "hanoi_landcover" else {}
    _, pngs, _ = _run(fn, task, S["units"], idx, key=key, hit=S["hit"], title=title, **extra)
    return pngs


def _detail(task, i):
    u = S["units"].iloc[i]
    kind = _kind(task)
    if kind == "ifs_manila":
        return _run(ifs_show, task, u.scene, box=(u.y0, u.x0, u["size"]))[1]
    if kind == "hanoi_landcover":
        return _run(lc_show, task, u.scene, target=S["target"], box=(u.y0, u.x0, u["size"]))[1]
    return _run(ai4_show, task, u.scene, y0=int(u.y0), x0=int(u.x0), size=int(u["size"]))[1]


def picks_view(max_shown=16):
    """The overlap table, the places each input would pick, its failures and one place in
    detail. Follows the settings of part 5."""
    if not _need_decision():
        return
    task = S["task"]
    labels = _short(task)
    shown = [k for k in task.inputs if task.has_image[k]]
    overlap = W.HTML()
    pick = W.ToggleButtons(options=[(labels[k], k) for k in task.inputs],
                           value="stage2" if "stage2" in task.inputs else task.inputs[-1],
                           description="Places picked with", style=_buttons("120px"))
    default_img = "stage1" if _kind(task) == "ai4_fields" else "stage2"
    on = W.ToggleButtons(options=[(labels[k], k) for k in shown], value=default_img,
                         description="Shown on the image of", style=_buttons("120px"))
    galleries = W.VBox([])
    which = W.ToggleButtons(options=[("a confident false alarm", "false"),
                                     ("a missed hit", "missed")], value="false",
                            description="Look closer at", style=_buttons("170px"))
    number = W.IntSlider(value=1, min=1, max=8, continuous_update=False, layout=_WIDE,
                         style=_LABEL, description="Number (1 = most striking)")
    detail = W.VBox([])
    status = W.HTML()

    def lists():
        score = np.atleast_2d(S["preds"][pick.value]).mean(0)
        hit, k = S["hit"], S["budget"]
        order = np.argsort(-score)
        chosen = set(order[:k])
        false = [i for i in order if not hit[i]][:8]
        missed = [i for i in np.argsort(-S["units"].truth.values)
                  if hit[i] and i not in chosen][:8]
        return top_k(score, k), false, missed

    def draw_detail(_=None):
        try:
            _, false, missed = lists()
            idx = false if which.value == "false" else missed
            if not idx:
                detail.children = (W.HTML(_note("None: nothing to show for this setting.")),)
                return
            number.max = len(idx)
            detail.children = _images(_detail(task, idx[min(number.value, len(idx)) - 1]))
        except Exception as exc:
            status.value = _error(exc)

    def draw(_=None):
        try:
            status.value = ""
            overlap.value = _html_table(overlap_table(S["preds"], S["budget"], labels=labels))
            picks, false, missed = lists()
            name, w = labels[pick.value], S["place_word"]
            note = (f" (the first {max_shown} of {len(picks)})" if len(picks) > max_shown else "")
            pngs = (_gallery(task, picks[:max_shown], on.value,
                             f"The {S['budget']} {w} you would visit using {name}{note}")
                    + _gallery(task, false, on.value,
                               f"{name}: highest-scoring {w} that are NOT hits")
                    + _gallery(task, missed, on.value, f"Clear hits that {name} would NOT visit"))
            galleries.children = _images(pngs)
            draw_detail()
        except Exception as exc:
            status.value = _error(exc)
    for w in (pick, on):
        w.observe(draw, names="value")
    for w in (which, number):
        w.observe(draw_detail, names="value")
    REFRESH.append(draw)
    ui = W.VBox([W.HTML("<b>Share of the picked places that two inputs have in common</b>"),
                 overlap, pick, on, status, galleries,
                 W.HTML("<br><b>One place in detail</b> (for the input picked above)"),
                 which, number, detail])
    ui.pick, ui.on, ui.which, ui.number, ui.galleries, ui.detail = (pick, on, which, number,
                                                                     galleries, detail)
    UI["picks"] = ui
    display(ui)
    draw()
    UI["finds"] = ask(("RESCUE", "A place where super-resolution changed the decision for the "
                       "better (which one, and why)", "text"),
                      ("FAILURE", "A failure worth telling the room about (which place, what went "
                       "wrong)", "text"))


# ---- part 8: the SR service ------------------------------------------------------------------

OWN_PLACE = "My own place"


def sr_panel(sr, ready_made, default_lat, default_lon):
    """Pick a ready-made area (or your own point), send it with one button and look at the
    result with sliders. The button waits until the job is done (3–6 minutes for a new area)."""
    from sr_client import SRError, area_around
    from sr_client.client import FINISHED
    place = W.Dropdown(options=[*ready_made, OWN_PLACE], description="Area", layout=_WIDE,
                       style=_LABEL)
    lat = W.FloatText(value=default_lat, description="Latitude", style=_LABEL)
    lon = W.FloatText(value=default_lon, description="Longitude", style=_LABEL)
    km = W.BoundedFloatText(value=2, min=0.5, max=5, step=0.5, description="Size (km)",
                            style=_LABEL)
    day = W.Text(value="2026-03-15", description="Date (YYYY-MM-DD)", style=_LABEL)
    own = W.VBox([W.HTML(_note("A point (in Google Maps: right-click, then click the "
                               "coordinates to copy them), a size and a date, ideally in the "
                               "dry season.")), lat, lon, km, day])
    own.layout.display = "none"
    stages = W.ToggleButtons(options=[("stage 1 and 2", "12"), ("stage 1 only (faster)", "1")],
                             value="12", description="Make", style=_buttons("170px"))
    send = W.Button(description="Send and wait", button_style="primary", icon="rocket",
                    layout=W.Layout(width="220px"))
    status = W.HTML()
    viewer = W.VBox([])

    def toggle(_=None):
        own.layout.display = "" if place.value == OWN_PLACE else "none"
    place.observe(toggle, names="value")

    def on_send(_):
        send.disabled = True
        viewer.children = ()
        try:
            if place.value == OWN_PLACE:
                area, d, options = area_around(lat.value, lon.value, km=km.value), \
                    day.value.strip(), {}
            else:
                p = ready_made[place.value]
                area, d, options = p["area"], p["date"], p.get("options", {})
            job = sr.submit(area, d, stages=list(stages.value), **options)
            t0 = time.monotonic()
            while job.status not in FINISHED:
                status.value = _pre(job._progress_line(time.monotonic() - t0))
                time.sleep(5)
                job.refresh()
            lines = [f"job {job.id[:8]}: {job.status} after "
                     f"{(time.monotonic() - t0) / 60:.1f} min"]
            lines += [f"warning [{w['code']}]: {w['message']}" for w in job.warnings]
            status.value = _pre("\n".join(lines))
            if job.status not in ("done", "partial"):
                status.value += _note(f"The job did not finish: {job.info.get('error')}",
                                      "#b00020")
                return
            S["job"] = job
            viewer.children = (_job_viewer(job),)
        except SRError as exc:
            status.value = _note(f"The service said: {_html.escape(exc.message)}", "#b00020")
        except Exception as exc:
            status.value = _error(exc)
        finally:
            send.disabled = False
    send.on_click(on_send)
    ui = W.VBox([place, own, stages, send, status, viewer])
    ui.place, ui.send, ui.status, ui.viewer, ui.own_fields = place, send, status, viewer, (
        lat, lon, km, day)
    UI["sr"] = ui
    display(ui)


def _rgb(da):
    a = np.moveaxis(np.nan_to_num(da.isel(band=[0, 1, 2]).values.astype("float32")), 0, -1)
    return a / max(float(a.max()), 1e-6) * 255


def _job_viewer(job):
    """Sentinel-2 and the SR stages side by side, with zoom and position sliders."""
    names = [("Sentinel-2 (10 m)", "lr_reference.tif"), ("SR stage 1", "stage1_sr.tif"),
             ("SR stage 2", "stage2_sr.tif")]
    files = job.results()["files"]
    layers = [(t, _rgb(job.open(f))) for t, f in names if f in files]
    h, w = layers[-1][1].shape[:2]
    px = 2.5
    half = int(min(h, w) * px / 2)
    size = W.ToggleButtons(options=[("250 m", 100), ("500 m", 200), ("1 km", 400),
                                    ("whole area", max(h, w))], value=min(400, max(h, w)),
                           description="Zoom", style=_buttons("90px"))
    east = W.IntSlider(value=0, min=-half, max=half, step=50, continuous_update=False,
                       description="← west | east → (m)", layout=_WIDE, style=_LABEL)
    north = W.IntSlider(value=0, min=-half, max=half, step=50, continuous_update=False,
                        description="← south | north → (m)", layout=_WIDE, style=_LABEL)
    image = W.Image(format="png", layout=W.Layout(max_width="100%"))

    def draw(_=None):
        s = min(size.value, h, w)
        cy = int(np.clip(h / 2 - north.value / px, s / 2, h - s / 2))
        cx = int(np.clip(w / 2 + east.value / px, s / 2, w - s / 2))
        fig, axes = plt.subplots(1, len(layers), figsize=(5 * len(layers), 5.3), squeeze=False)
        for ax, (t, a) in zip(axes[0], layers):
            lh, lw = a.shape[:2]
            f = lh / h      # the 10 m image has fewer pixels
            crop = a[int((cy - s / 2) * f):int((cy + s / 2) * f),
                     int((cx - s / 2) * f):int((cx + s / 2) * f)]
            ax.imshow(autostretch(crop), interpolation="nearest")
            ax.set_title(t), ax.set_xticks([]), ax.set_yticks([])
        plt.tight_layout()
        image.value = _png(fig)
    for c in (size, east, north):
        c.observe(draw, names="value")
    draw()
    box = W.VBox([size, east, north, image])
    box.size, box.east, box.north, box.image = size, east, north, image
    return box


# ---- part 9: the numbers for your slide ------------------------------------------------------

def card_text():
    task = S["task"]
    lines = ["=" * 72, f"TASK      {task.meta['title']}"]
    for k, v in scenario_dict().items():
        lines.append(f"{k.replace('_', ' ').upper():<22}{v}")
    lines += ["-" * 72, "SETTINGS  " + ", ".join(f"{k} {v}" for k, v in S["settings"].items()),
              "", S["res"][["hits", "range over models", "hit rate", "missed places",
                             "cost"]].to_string(), "-" * 72]
    for key, text in (("VERDICT", "verdict"), ("CHANGE_MIND", "what would change our mind"),
                      ("RESCUE", "where SR helped"), ("FAILURE", "a failure")):
        v = ANSWERS.get(key)
        lines.append(f"{text + ':':<28}{v if v not in ('', None) else '(not answered)'}")
    lines.append("=" * 72)
    return "\n".join(lines)


def slide_card(fname="my_slide_chart.png"):
    """Your verdict, then a card with everything the slide needs; it follows the decision
    settings and your answers. The button downloads the chart."""
    if not _need_decision():
        return
    UI["verdict"] = ask(("VERDICT", "Your verdict", ["deploy", "pilot", "don't use"]),
                        ("CHANGE_MIND", "What would change your mind?", "text",
                         "a validation on 50 blocks in our own city"))
    card = W.HTML()
    chart = W.Image(format="png", layout=W.Layout(max_width="100%"))
    button = W.Button(description=f"Download the chart ({fname})", icon="download",
                      layout=W.Layout(width="320px"))
    note = W.HTML()

    def update(_=None):
        card.value = _pre(card_text())
        fig = plot_decision(S["res"], S["budget"], scenario_dict()["decision"])
        fig.savefig(fname, dpi=200, bbox_inches="tight")
        chart.value = _png(fig)

    def download(_):
        update()
        try:
            from google.colab import files
            files.download(fname)
            note.value = _note("Downloading ...", "#1a7f37")
        except Exception:  # not in Colab
            note.value = _note(f"Saved as {fname} next to this notebook.", "#1a7f37")
    button.on_click(download)
    REFRESH.append(update)
    ANSWER_REFRESH.append(update)
    update()
    ui = W.VBox([card, chart, button, note])
    ui.card, ui.chart, ui.button = card, chart, button
    UI["card"] = ui
    display(ui)

In [ ]:
import os

DATA_URL = os.environ.get("HK_DATA_URL", "https://huggingface.co/datasets/adbsr-internal/ssta-hackathon/resolve/main")

task = get_task("ifs_manila", DATA_URL, extras=('ifs_manila_hr',))
labels = {k: SHORT[k] for k in task.inputs}
task

## 1. The data

Experts outlined informal settlements on real high-resolution images (Google satellite imagery,
about 0.5 m). The tiles were chosen to contain informal settlements of different sizes, so
settlements are much more common here than in the city as a whole.

Six task models were trained on the real high-resolution images (resampled to 2.5 m). They have
**never seen a super-resolved image**: we simply give them each input and see what they find.
This is the "reuse a model you already have" path from Session 3.

**The inputs.** Every place in this dataset comes in up to four versions, and the same task
model is applied to each:

| Name in this notebook | What it is |
|---|---|
| **Sentinel-2** | the free 10 m image, enlarged to 2.5 m pixels by plain interpolation (bicubic). This is the "do nothing" option. |
| **SR stage 1** | our model's first stage: combines 8 Sentinel-2 images into one sharper 2.5 m image. Conservative: it does not invent texture. |
| **SR stage 2** | the full model: stage 1 plus the diffusion refiner (τ = 250), which adds realistic texture. Sharper, but some detail is made up. |
| **Real high-res** | the real commercial image resampled to 2.5 m: the ceiling, what you would get if you bought imagery. |

The real high-resolution images are licensed for this workshop only. If they did not
download, everything still works: you just will not see those pictures.

In [ ]:
# The tile with the most informal settlement: images (top) and what the models flag (bottom)
example = task.scenes.informal_share.idxmax()
ifs_show(task, example)

## 2. How good is the model on each input?

The usual way to judge a map: pixel by pixel against the expert outlines. **F1** combines false
alarms and misses: 0 % = nothing right, 100 % = perfect. "Lowest" and "highest" show the spread
over the six task models; "paper" is the value in our paper, which scores overlapping patches
and so differs slightly.

In [ ]:
ifs_f1_table(task)

**What this says.** On plain Sentinel-2 the models find almost nothing; on super-resolved images
they find about half of what they find on real high-resolution images. Stage 2, with its
synthesised roof texture, does best. But a pixel score is not a decision. The rest of the
notebook asks whether these differences change **where you would send people**.

## 3. Look around, including failures

Choose a tile in the menu (sorted from most to least informal settlement); the picture follows
at once. Top row: what the model sees. Bottom row: orange = flagged as informal (pixels where the
six models' mean probability is above the threshold, 0.5 unless you move the slider), black
line = expert outline. You can also look at one model at a time. Look for:
* places flagged on super-resolved images but not on real images: **invented** settlements?
* settlements missed everywhere: what do they look like?

In [ ]:
explore(task)

## 4. Your scenario

Before you look at any decision numbers, agree as a team on **a real decision** this map would
feed. Be concrete; the numbers in part 5 only mean something for a concrete decision.

* **Who decides**, and what do they decide?
* **Budget**: how many places can they act on (visits, audits, teams, km² of map updates)?
* **What counts as a hit**: when is a place worth the visit?
* **What does a wasted visit cost**, and **what does a missed place cost**? Money, days, people,
  or harm. Rough is fine; the ratio matters most.

Type your answers into the fields: they are saved as you type and end up on your slide card in
part 9. The grey text is an example; if you have no idea of your own, the button copies the
example into the empty fields, and you change what does not fit your agency.

In [ ]:
scenario(task)

## 5. The decision cell

The map is cut into square blocks. For every block and every input, the models' map gives a
score: the share of the block flagged as informal. You visit the **highest-scoring blocks**, up
to your budget. Afterwards, the expert outlines tell us which visits were hits.

Set the controls to match your scenario: the chart and the table follow at once, and so do
parts 6 and 9.

In [ ]:
decision_panel(task)

**How to read it.** *Hits* is the mean over the six task models; *range* shows the worst and
best model, a reminder that a single model can be lucky or unlucky. *Random* is what you would
expect without any map; *perfect* is the best your budget allows. *Cost* = wasted visits ×
their cost + missed places × their cost, in your units.

Does the advantage hold for a smaller or bigger budget? (This chart follows part 5 too.)

In [ ]:
sweep_view()

## 6. Did super-resolution change *which* places you picked?

A better score does not always mean a different decision. The table shows how many of the
places picked with one input are also picked with another (100 % = the same list). Then look at
the places themselves: green frames are hits, red frames are wasted visits. Choose the input
whose picks you want to see; below them are its most confident false alarms and the clear hits
it would not visit. Everything here follows the settings of part 5.

In [ ]:
picks_view()

## 7. Optional: improve the baseline

Only if you have time, and only if it helps your decision. Ideas:
* **Threshold**: `ifs_units(task, cell_m=100, thr=0.3)` flags more pixels. Does a lower
  threshold help the Sentinel-2 or stage 1 inputs? (worked example in the next cell)
* **Score**: rank blocks by the mean probability instead of the share above the threshold
  (the arrays are in `task.arr(tile, "prob_stage2")`, values 0–255, one slice per model).
* **One model or six?** Use `preds[key][m]` for model `m` and see how much the decision depends
  on the model you happen to have.
* **Combine inputs**: e.g. visit a block only if both stage 1 and stage 2 flag it.

`current()` gives the decision of part 5 as it is set now; `decide`, `plot_decision` and
`sweep` work on anything of the same shape.

In [ ]:
units, preds, hit, res = current()   # the decision of part 5, as it is set now

# Worked example: a lower threshold (0.3) with the same block size, hit rule and budget
block_m = int(units["size"].iloc[0] * task.px)
units_low, preds_low = ifs_units(task, cell_m=block_m, thr=0.3)
hit_low = units_low.truth.values >= S["hit_at"]
decide(units_low, preds_low, hit_low, S["budget"], S["cost_wasted"], S["cost_missed"], labels=labels)

## 8. Optional: super-resolve an area of your own

With the SR service you can make stage 1 and stage 2 images for any area up to 5 km × 5 km.
Your API key (`srk_...`) and the service address are on your team card; in Colab, store them
under **Secrets** (key icon on the left) as `SR_API_KEY` and `SR_API_URL`, or paste them when
asked. The ready-made areas were run before the workshop, so they come back in seconds and cost
nothing. A new area takes 3–6 minutes, and the button waits until it is done. You can add your
own areas to `READY_MADE`, or choose "My own place" in the menu.

There is no informal-settlement model in this part (its weights are not public). Use it to look: would *you* spot the settlements you know on stage 1 or stage 2?

In [ ]:
%pip install -q "sr-client[geo,viz] @ git+https://github.com/adb-sr/sr-client@v0.1.0"

In [ ]:
import getpass

from sr_client import SRClient, area_around


def secret(name, prompt):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return getpass.getpass(prompt)


sr = SRClient(secret("SR_API_KEY", "API key (srk_...): "), url=secret("SR_API_URL", "Service URL: "))
sr.me()["budget"]

In [ ]:
READY_MADE = {
    # name: area, date and options; run before the workshop, so they come back fast
    "Tondo and Baseco, Manila": dict(area=area_around(14.5960, 120.9620, km=3), date="2026-03-15"),
    "Payatas, Quezon City": dict(area=area_around(14.7130, 121.1000, km=3), date="2026-03-15"),
    "Manila South test area": dict(area=area_around(14.5560, 121.0136, km=2), date="2025-06-01"),
}

In [ ]:
sr_panel(sr, READY_MADE, default_lat=14.596, default_lon=120.962)

## 9. Numbers for your slide

Choose your verdict below. The card collects your scenario (part 4), the decision settings
(part 5) and your answers; it follows any change by itself. The button downloads the chart.

**Your slide** (one slide, template from the facilitators) has four parts:
1. **Scenario**: who decides, budget, what a hit is, what errors cost.
2. **Numbers**: hits for Sentinel-2, SR stage 1, SR stage 2 (and real high-res if you have it).
3. **Verdict**: *deploy*, *pilot* or *don't use*, and for which decision.
4. **What would change your mind**: data, validation, partners, costs.

Questions to discuss before you write the verdict:
* Would you act on the map directly, or only use it to decide where to look?
* Who checks the output, and what would make them stop trusting it?
* What is missing to run this for real: data, validation, partners?

In [ ]:
slide_card()

---
Contains modified Copernicus Sentinel data. Super-resolution: ADB SSTA model (stage 1 Sen4x,
stage 2 diffusion refiner). Workshop material: please do not redistribute the data packages.

Informal-settlement outlines: ADB technical assistance, expert annotators. Real high-resolution images: Google satellite imagery, workshop use only.